# Explainable Diabetes Risk Prediction — Model Training & Evaluation

This notebook implements the complete machine learning stage comparing:
1. **Random Forest Classifier**
2. **XGBoost Classifier**

Methodology:
- Leak-free stratified 80/20 train-test split (`random_state=42`)
- 5-Fold Stratified Cross-Validation for hyperparameter tuning on training data
- Single-pass final test set evaluation
- Rigorous multi-metric comparison (Accuracy, Precision, Recall, F1-Score, ROC-AUC)
- Feature importance extraction and artifact generation for SHAP explainability

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ml.model_utils import load_processed_data, split_data, FEATURE_COLUMNS, TARGET_COLUMN
from ml.train_models import tune_random_forest, tune_xgboost, cross_validate_models
from ml.evaluate_models import evaluate_and_save_pipeline

print("Libraries successfully imported.")

## 1. Load Data & Perform Leak-Free Stratified Split

In [ ]:
data = load_processed_data(PROJECT_ROOT / 'data' / 'diabetes.csv')
X_train, X_test, y_train, y_test = split_data(data, test_size=0.20, random_state=42)

print(f"Total samples: {len(data)}")
print(f"Training set: {X_train.shape[0]} samples, Target distribution:\n{y_train.value_counts()}")
print(f"Testing set:  {X_test.shape[0]} samples, Target distribution:\n{y_test.value_counts()}")

## 2. Hyperparameter Tuning on Training Set (5-Fold Stratified CV, F1 Optimization)

In [ ]:
# Tune Random Forest
best_rf, rf_params, rf_score = tune_random_forest(X_train, y_train, random_state=42)
print(f"Random Forest Best F1: {rf_score:.4f}")
print(f"Best RF Hyperparameters: {rf_params}\n")

# Tune XGBoost
best_xgb, xgb_params, xgb_score = tune_xgboost(X_train, y_train, random_state=42)
print(f"XGBoost Best F1: {xgb_score:.4f}")
print(f"Best XGBoost Hyperparameters: {xgb_params}")

candidate_models = {
    'Random Forest': best_rf,
    'XGBoost': best_xgb
}

## 3. Stratified 5-Fold Cross-Validation Performance

In [ ]:
cv_df = cross_validate_models(candidate_models, X_train, y_train, cv_splits=5, random_state=42)
cv_df

## 4. Final Evaluation on Untouched Test Set & Visual Artifact Generation

In [ ]:
output_dirs = [PROJECT_ROOT / 'results', PROJECT_ROOT / 'ml' / 'results']
results = evaluate_and_save_pipeline(
    selected_models=candidate_models,
    cv_results_df=cv_df,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    output_dirs=output_dirs
)

print(f"Selected Best Model: {results['best_model_name']}\n")
results['comparison_df']

## 5. Preview Generated Visualizations

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(PROJECT_ROOT / 'results' / 'roc_curve_comparison.png')))
display(Image(filename=str(PROJECT_ROOT / 'results' / 'metric_comparison.png')))
display(Image(filename=str(PROJECT_ROOT / 'results' / 'random_forest_feature_importance.png')))
display(Image(filename=str(PROJECT_ROOT / 'results' / 'xgboost_feature_importance.png')))